# CMP6202 – Notebook 01: Data preparation

Evidence for report sections **1.1** and **3.3**. Reads the raw 14 February 2018 file, cleans and de-duplicates it,
and saves the 75,000-flow sample that notebooks 02–04 load.

Research questions answered across the four notebooks:
* **RQ1:** without duplicate flows and label proxies, how well do eight classifiers separate both brute-force attacks
  from benign traffic?
* **RQ2:** do the models learn attack behaviour or testbed artefacts (port, attackers' TCP settings, an unseen day)?
* **RQ3:** with only 46 distinct FTP flows, does SMOTE beat class weights, Tomek links or no resampling?

In [1]:
# Google Colab setup. On a local machine this cell does nothing.
# On Colab: upload the project folder to Google Drive (README, "Run on Google Colab"). The cell mounts Drive,
# finds the folder, installs the pinned libraries and makes `nids` importable.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    from importlib.metadata import version

    from google.colab import drive

    drive.mount("/content/drive")
    # I accept both upload locations the README describes and pick the one that holds the nids package.
    candidates = ["/content/drive/MyDrive/CMP6202_NIDS/Project", "/content/drive/MyDrive/Project"]
    found = [path for path in candidates if os.path.isdir(os.path.join(path, "src", "nids"))]
    if not found:
        raise SystemExit(f"Project folder not found on Drive; looked in {candidates}")
    PROJECT_DIR = found[0]
    os.chdir(PROJECT_DIR)
    loaded = {name: version(dist) for name, dist in (("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"))
              if name in sys.modules}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
    changed = [name for name, old in loaded.items()
               if old != version({"sklearn": "scikit-learn"}.get(name, name))]
    if changed:
        # A library that was already imported cannot be replaced in a running session.
        raise SystemExit(f"Upgraded {changed}: choose Runtime > Restart session, then run this cell again.")
    print("Running on Colab from", PROJECT_DIR)
else:
    print("Running locally; Colab setup skipped")

Running locally; Colab setup skipped


## **Import the libraries**

In [2]:
from time import perf_counter

import pandas as pd
from IPython.display import display

from nids import config, data, report_numbers  # my package: src/nids

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1.1 Dataset identification

CSE-CIC-IDS2018 (CSE and CIC, AWS testbed), Kaggle mirror under **CC BY 4.0**. Each row is one CICFlowMeter-V3 flow.

In [3]:
columns = data.raw_columns()
print(f"File: {config.RAW_CSV.name} ({config.RAW_CSV.stat().st_size / 1e6:,.1f} MB), {len(columns)} columns")
print(f"Target: '{config.LABEL_COL}'; non-feature columns: 'Dst Port', 'Protocol', 'Timestamp'")

File: 02-14-2018.csv (358.2 MB), 80 columns
Target: 'Label'; non-feature columns: 'Dst Port', 'Protocol', 'Timestamp'


# 3 Experimental Design
## 3.3 Data cleaning and pre-processing transformations

Cleaning works row by row and learns nothing from the data, so it can run before the split without leaking test
information. Scaling and SMOTE are fitted later, inside the cross-validated pipeline.

### (a) Load, clean and de-duplicate

In [4]:
# I read the 358 MB file in chunks of 200,000 rows with float64 types (data.build_unique_flows).
start = perf_counter()
flows, report = data.build_unique_flows()
print(f"Read, cleaned and de-duplicated {report.raw.sum():,} rows in {perf_counter() - start:.1f} s")

Read, cleaned and de-duplicated 1,048,575 rows in 3.9 s


### (b) Non-finite and impossible values

Zero-duration flows give `Infinity`/`NaN` rates; I drop them rather than invent a rate. Negative durations are clock
errors. The value `-1` in the TCP window columns means “no window observed”, so I keep it.

In [5]:
problems = pd.DataFrame({"Infinity": report.inf_by_column, "NaN": report.nan_by_column,
                         "Negative": report.negative_by_column}).fillna(0).astype(int)
problems = problems[(problems > 0).any(axis=1)]
problems.to_csv(config.TABLES_DIR / "3_3_invalid_values_by_column.csv")
problems

,Infinity,NaN,Negative
Flow Byts/s,1547,2277,0
Flow Duration,0,0,5
Flow IAT Max,0,0,2
Flow IAT Mean,0,0,5
Flow IAT Min,0,0,5
Flow Pkts/s,3824,0,5
Fwd IAT Max,0,0,2
Fwd IAT Mean,0,0,5
Fwd IAT Min,0,0,5
Fwd IAT Tot,0,0,5


### (c) Duplicate flows and contradictory labels

Two rows are the same flow when every feature except `Timestamp` and `Dst Port` matches. Without this step a random
split would put copies of one flow in both training and test. Vectors labelled both FTP and SSH are removed.

In [6]:
display(report.stage_table())
print(f"Contradictory vectors removed: {report.n_conflicting_vectors} "
      f"({int(report.conflicts_removed.sum())} rows)")

Label,Benign,FTP-BruteForce,SSH-Bruteforce,Total
1. Raw file,667626,193360,187589,1048575
2. Removed: non-finite values (Infinity/NaN),3818,6,0,3824
3. Removed: impossible negative values,5,0,0,5
4. After row-wise cleaning,663803,193354,187589,1044746
5. Removed: duplicate flows (same features and label),138563,193301,93541,425405
6. Removed: same features with different labels,0,7,7,14
7. Unique flows (sampling frame),525240,46,94041,619327


Contradictory vectors removed: 7 (14 rows)


### (d) Stratified sample of 75,000 flows

Stratified by `Label`, but every class keeps at least `min(class size, 2,000)` rows, so all unique FTP flows stay in.

In [7]:
sample = data.stratified_sample(flows)
distribution = pd.DataFrame({
    "Raw rows": report.raw,
    "Raw %": report.raw / report.raw.sum() * 100,
    "Unique flows": report.unique,
    "Unique %": report.unique / report.unique.sum() * 100,
    "Sample rows": sample[config.LABEL_COL].value_counts(),
    "Sample %": sample[config.LABEL_COL].value_counts(normalize=True) * 100,
}).sort_index()
distribution.loc["Total"] = distribution.sum()
distribution.round(3).to_csv(config.TABLES_DIR / "3_3_label_distribution.csv")
distribution.round(3)

,Raw rows,Raw %,Unique flows,Unique %,Sample rows,Sample %
Label,,,,,,
Benign,667626,63.67,525240,84.808,63572,84.763
FTP-BruteForce,193360,18.44,46,0.007,46,0.061
SSH-Bruteforce,187589,17.89,94041,15.184,11382,15.176
Total,1048575,100.00,619327,100.000,75000,100.000


In [8]:
# RQ3 quotes the number of distinct FTP flows, so I stop here if the data no longer agrees.
assert report.unique["FTP-BruteForce"] == 46, "the distinct FTP count changed: update RQ3"

### Attribute table (Appendix A of the report)

Each column with its CIC description, level of measurement and whether it is constant (constant columns are dropped
later by `VarianceThreshold`).

In [9]:
attributes = data.attribute_table(flows)
# CIC's wording where it exists, otherwise my own note, marked [Author].
has_cic_text = attributes["cic_description"].notna() & attributes["cic_description"].astype(str).str.strip().ne("")
attributes["report_description"] = attributes["cic_description"].where(has_cic_text, "[Author] " + attributes["note"].astype(str))
attributes.to_csv(config.TABLES_DIR / "1_1_attributes.csv", index=False)

levels = attributes["level_of_measurement"].value_counts()
levels.to_csv(config.TABLES_DIR / "1_1_levels_of_measurement.csv")
display(levels.rename("columns").to_frame())
attributes[["csv_column", "role", "level_of_measurement", "dtype", "n_unique", "constant"]]

,columns
level_of_measurement,
ratio (discrete),52
ratio (continuous),24
nominal,3
interval,1


,csv_column,role,level_of_measurement,dtype,n_unique,constant
0,Dst Port,categorical,nominal,float64,7770,False
1,Protocol,categorical,nominal,float64,3,False
2,Timestamp,identifier,interval,string,31919,False
3,Flow Duration,numeric,ratio (discrete),float64,389487,False
4,Tot Fwd Pkts,numeric,ratio (discrete),float64,750,False
...,...,...,...,...,...,...
75,Idle Mean,numeric,ratio (continuous),float64,80416,False
76,Idle Std,numeric,ratio (continuous),float64,57844,False
77,Idle Max,numeric,ratio (discrete),float64,59131,False
78,Idle Min,numeric,ratio (discrete),float64,69043,False


### Save the sample and its manifest (SHA-256, seed, stage counts, library versions)

In [10]:
manifest = data.save_sample(sample, report)
final_stages = report.stage_table(sample[config.LABEL_COL].value_counts())
final_stages.to_csv(config.TABLES_DIR / "3_3_cleaning_stages.csv")
print(f"Saved {config.SAMPLE_PARQUET.name}, sha256 {manifest['sample_sha256'][:16]}…")

Saved sample_75k_dedup.parquet, sha256 34cc46c3488eab82…


## **Save the numbers quoted in the report** (`reports/numbers.json`)

In [11]:
raw, unique = report.raw, report.unique
counts = sample[config.LABEL_COL].value_counts()
report_numbers.record(
    raw_rows=f"{raw.sum():,}",
    raw_benign=f"{raw['Benign']:,}", raw_ftp=f"{raw['FTP-BruteForce']:,}", raw_ssh=f"{raw['SSH-Bruteforce']:,}",
    nonfinite_removed=f"{int(report.nonfinite_removed.sum()):,}",
    negative_removed=f"{int(report.negative_removed.sum()):,}",
    duplicates_removed=f"{int(report.duplicates_removed.sum()):,}",
    duplicates_share=f"{report.duplicates_removed.sum() / report.after_cleaning.sum():.1%}",
    ftp_repeat_share=f"{1 - unique['FTP-BruteForce'] / raw['FTP-BruteForce']:.2%}",
    unique_ftp=f"{unique['FTP-BruteForce']}", unique_ssh=f"{unique['SSH-Bruteforce']:,}",
    unique_benign=f"{unique['Benign']:,}", unique_total=f"{unique.sum():,}",
    conflicting_vectors=f"{report.n_conflicting_vectors}",
    conflicting_rows=f"{int(report.conflicts_removed.sum())}",
    sample_size=f"{config.SAMPLE_SIZE:,}",
    sample_benign=f"{counts['Benign']:,}", sample_ftp=f"{counts['FTP-BruteForce']}",
    sample_ssh=f"{counts['SSH-Bruteforce']:,}",
)

{'raw_rows': '1,048,575',
 'raw_benign': '667,626',
 'raw_ftp': '193,360',
 'raw_ssh': '187,589',
 'nonfinite_removed': '3,824',
 'negative_removed': '5',
 'duplicates_removed': '425,405',
 'duplicates_share': '40.7%',
 'ftp_repeat_share': '99.98%',
 'unique_ftp': '46',
 'unique_ssh': '94,041',
 'unique_benign': '525,240',
 'unique_total': '619,327',
 'conflicting_vectors': '7',
 'conflicting_rows': '14',
 'sample_size': '75,000',
 'sample_benign': '63,572',
 'sample_ftp': '46',
 'sample_ssh': '11,382'}